## Задание 1

In [1]:
import pandas as pd

In [2]:
cols_to_use = [
    'order_id', 'customer_id', 'order_status',
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df_orders = pd.read_csv('olist_orders_dataset.csv', usecols = cols_to_use)

date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df_orders[date_cols] = df_orders[date_cols].apply(pd.to_datetime, errors = 'coerce')

mem_before = df_orders['order_status'].memory_usage(deep = True)
print(f'память до (object): {mem_before} байт')
df_orders['order_status'] = df_orders['order_status'].astype('category')
mem_after = df_orders['order_status'].memory_usage(deep = True)
print(f'память после (category): {mem_after} байт')
print(f'уникальных статусов: {df_orders["order_status"].nunique()}')

память до (object): 5766064 байт
память после (category): 100033 байт
уникальных статусов: 8


**Почему order_status выгодно хранить как category**

В столбце почти 100 тысяч строк, но уникальных статусов всего 8 (delivered, shipped, canceled и т.д.). В типе object каждая ячейка хранит ссылку на отдельный питоновский объект строки, поэтому одни и те же слова лежат в памяти десятки тысяч раз. Тип category хранит словарь из 8 уникальных значений один раз, а в самих ячейках лежат маленькие целочисленные коды (int8). За счет этого объем памяти падает в десятки раз, а сравнения, фильтрация и группировка по статусу работают быстрее, потому что сравниваются числа, а не строки.

## Задание 2

In [3]:
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')

review_date_cols = ['review_creation_date', 'review_answer_timestamp']
df_reviews[review_date_cols] = df_reviews[review_date_cols].apply(pd.to_datetime, errors = 'coerce')

In [4]:
df_reviews['review_comment_title'] = df_reviews['review_comment_title'].fillna('No title')
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna('No message')

In [5]:
text_cols = ['review_comment_title', 'review_comment_message']

for col in text_cols:
  df_reviews[col] = (
      df_reviews[col]
      .str.lower()
      .str.replace(r'[\n\r]+', ' ', regex = True)
      .str.strip()
  )

In [6]:
print(f'дубликатов по order_id: {df_reviews["order_id"].duplicated().sum()}')

df_reviews = (
    df_reviews
    .sort_values('review_creation_date', ascending = False)
    .drop_duplicates(subset = 'order_id', keep = 'first')
    .reset_index(drop = True)
)

дубликатов по order_id: 551


In [7]:
print(df_reviews.isna().sum())
print(df_reviews['order_id'].duplicated().sum())
print(df_reviews.dtypes)
print(df_orders.dtypes)
print(f'пропусков в order_status: {df_orders["order_status"].isna().sum()}')

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64
0
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object
order_id                                    str
customer_id                                 str
order_status                           category
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object
пропусков в order_status: 0


## Задание 3

почтовый индекс читаем как строку, тк если pandas прочитает его как int64, ведущий ноль пропадет (01151 превратится в 1151), и фильтр по префиксу "01" перестанет работать

In [8]:
df_customers = pd.read_csv(
    'olist_customers_dataset.csv',
    dtype = {'customer_zip_code_prefix': str}
)
df_customers['city_initial'] = df_customers['customer_city'].str[0].str.upper()
df_customers[['customer_city', 'city_initial']].head()

,customer_city,city_initial
0,franca,F
1,sao bernardo do campo,S
2,sao paulo,S
3,mogi das cruzes,M
4,campinas,C


In [9]:
mask = (
    (df_customers['customer_state'] == 'SP') & (df_customers['customer_zip_code_prefix'].str.startswith('01'))
)

sp_customers = df_customers.loc[mask]
print(f'найдено клиентов: {len(sp_customers)}')
sp_customers.head()

найдено клиентов: 1716


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,city_initial
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,S
292,d703cdda08125e2b263b92746af0f23c,4669d94e1e35f2493af76862ff85beba,01527,sao paulo,SP,S
300,dcb6e7bc4e65d815025d0aa14349a4ca,a1864cdd58debf5031958069ee937742,01230,sao paulo,SP,S
333,ee23d61fedfc4227bb0e188654c28d08,494c6658198b47787205a06421bb9403,01321,sao paulo,SP,S
364,3f5939e06e89efd9e9ca8ef3ab9d68c7,6b7e1090d106aaee31d5ed12d2bbc2af,01310,sao paulo,SP,S


In [10]:
df_customers = df_customers.set_index('customer_id')
df_customers.head()

,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,city_initial
customer_id,,,,,
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,F
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,S
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,S
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,M
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,C


In [11]:
example_id = '4e7b3e00288586ebd08712fdd0374a03'
df_customers.loc[example_id]

customer_unique_id          060e732b5b29e8181a18229c7b0b2b5e
customer_zip_code_prefix                               01151
customer_city                                      sao paulo
customer_state                                            SP
city_initial                                               S
Name: 4e7b3e00288586ebd08712fdd0374a03, dtype: str